# The surface spec, `forward()`, and `linearize()`

`SurfaceSpec` is the declarative Bayesian response surface: treatments with kernels and
carryover, an outcome, nuisance terms, an intercept structure, interactions, a likelihood.
`build()` turns it into a `ModelSpec`; `Surface` wraps it as a `SupportsForward`. `forward`
is the value interpreter on the tree, and `linearize` gives the design matrix in the *linear*
parameters at a fixed point of the nonlinear ones — with the 1e-12 invariant that gate 9 checks.

In [ ]:
import numpy as np

from axiom.core import D, Likelihood, Outcome, Treatment, dimension, latex
from axiom.surface import (
    EventIndicators, FourierSeasonality, GeometricCarryover, HillKernel, InterceptKind, LinearTrend,
    NuisanceSet, ParameterRole, Surface, SurfaceSpec, build, check_linearization, column_names,
    design_matrix, forward, interaction_name, linear_coefficients, parameter_roles, prepare,
)
from axiom.sim import surface_world

In [ ]:
kind: InterceptKind = "hierarchical"
spec = SurfaceSpec(
    name="two_treatment_panel",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"), Treatment(name="b", dimension=D.currency, unit="USD")),
    outcome=Outcome(name="y", dimension=D.outcome, unit="units"),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
    carryover={"a": GeometricCarryover(max_lag=4)},
    nuisance=NuisanceSet(terms=(FourierSeasonality(period=12.0, order=1),)),
    intercept=kind,
    interactions=(("a", "b"),),
    likelihood=Likelihood(family="normal", scale="sigma"),
    unit_labels=("u0", "u1", "u2"),
)
model = build(spec)
roles: dict[str, ParameterRole] = parameter_roles(spec)
print(sorted(p.name for p in model.parameters))
print({k: v for k, v in roles.items()})
print(interaction_name("a", "b"), "|", dimension(model.mean))

In [ ]:
surface = Surface(spec)
print(surface.linear, "|", surface.nonlinear, "|", surface.auxiliary)
print(latex(surface.expr)[:160], "...")

## Nuisance terms

`NuisanceTerm` is the protocol (`AnyNuisanceTerm` the union of the shipped terms). A `LinearTrend`
with no explicit origin/scale is resolved against the fitted panel, and `resolve_conventions`
records what was used so a *forecast* panel gets the same basis.

In [ ]:
from axiom.surface import AnyNuisanceTerm, GRID_VERSION, LinearTrend, NuisanceTerm, resolve_conventions

term: NuisanceTerm = LinearTrend()
any_term: AnyNuisanceTerm = FourierSeasonality(period=12.0, order=1)
print(term.column_names("n0_"), any_term.column_names("n1_"))
trend_spec = spec.model_copy(update={"nuisance": NuisanceSet(terms=(LinearTrend(),)), "carryover": {}})
world_t = surface_world(n_units=3, n_periods=12, treatments=("a", "b"), nuisance=NuisanceSet(terms=(LinearTrend(),)), seed=2)
conv = resolve_conventions(world_t.spec, world_t.panel)
print(conv)
print("predict marks a dose grid with version", repr(GRID_VERSION))

## Panel layout

`prepare()` turns a role-tagged `Panel` into the `(n_units, n_periods)` arrays the tree reads —
time last, so `Convolve` acts within each unit and never across units. It refuses unbalanced
panels, unit mismatches, and non-equispaced periods when carryover is declared.

In [ ]:
world = surface_world(n_units=3, n_periods=24, treatments=("a", "b"), carryover={"a": GeometricCarryover(max_lag=4)}, seed=1)
data = prepare(world.spec, world.panel)
print({k: v.shape for k, v in data.items()})

## `forward()` and the linearization invariant

At fixed nonlinear parameters the mean is linear in the amplitudes, intercepts, interaction
and nuisance coefficients. `linearize` builds `X` column by exact unit-vector evaluation of
the tree, and `check_linearization` reports `‖X @ theta_lin + offset − forward‖∞`.

In [ ]:
ws = Surface(world.spec)
mu = forward(ws, world.data, world.theta)
dm = ws.linearize(world.data, world.theta)
print(mu.shape, dm.X.shape, dm.columns)
print("invariant:", check_linearization(ws, world.data, world.theta))
print(column_names(ws.model, ws.linear)[:4], linear_coefficients(dm.columns, world.theta)[:4].round(3))
print(np.allclose(dm.predict(world.theta).reshape(mu.shape), mu))
print(design_matrix(ws.model, ws.linear, world.data, world.theta).at.keys())

## Steady state for row grids

A surface with carryover interprets the last axis as time. To evaluate independent candidate
allocations as rows — what the allocator and optimal designs do — use `steady_state()`: the same
spec with no carryover, valid because the weights sum to one.

In [ ]:
grid = {"a": np.array([0.0, 50.0, 100.0]), "b": np.array([10.0, 10.0, 10.0]), "unit": np.array([0, 0, 0]), **{c: np.zeros(3) for c in ("n0_sin_1", "n0_cos_1")}}
try:
    ws.forward(grid, world.theta)
except ValueError as e:
    print("refused:", str(e)[:100])
print(ws.steady_state().forward(grid, world.theta).round(3))